# 05 · Forecasting Aggregate Clinic Activity

Lecture 1 — From Messy Medical Data to Statistical Insight

*Created by Master of AI.*

> **Synthetic educational data only.** This training is not intended for diagnosis, treatment, triage, medication decisions, or clinical decision-making.

We forecast **aggregate operational volumes** for planning — never individual disease progression.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
DATA = "../data/"
ts = pd.read_csv(DATA + "monthly_clinic_activity.csv", parse_dates=["month"]).set_index("month")
ts.tail()

In [ ]:
import matplotlib.pyplot as plt
ts.plot(figsize=(10, 4), title='Monthly activity (synthetic)'); plt.show()

## 1. Decomposition: trend + seasonality + remainder

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
dec = seasonal_decompose(ts["screening_visits"], model="additive", period=12)
dec.plot(); plt.show()

## 2. Hold-out evaluation (last 12 months)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
y = ts["screening_visits"].astype(float)
train, test = y[:-12], y[-12:]
naive = np.array([train.iloc[-12 + i] for i in range(12)])
hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit().forecast(12)
def scores(a, p):
    e = np.asarray(a) - np.asarray(p)
    return {"MAE": np.abs(e).mean().round(2), "RMSE": np.sqrt((e**2).mean()).round(2),
            "MAPE %": (100 * np.abs(e / np.asarray(a))).mean().round(2)}
print("seasonal naive", scores(test, naive))
print("Holt-Winters  ", scores(test, hw))

## 3. Forecast the next 12 months

In [ ]:
model = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=12).fit()
fc = model.forecast(12)
ax = y.plot(figsize=(10, 4), label='history'); fc.plot(ax=ax, label='forecast'); ax.legend(); plt.show()
fc.round(0)

**Limitations:** forecasts assume the past pattern continues; policy changes, outbreaks or new services break that assumption. Always communicate uncertainty ranges.